# Homework 1: Due on 11/1 11.59pm

**Total: 40 points.** Every problem is an implementation task, self-checked by running the cell.

| # | You implement | Pts |
|---|---|---:|
| 1 | The regularized MLP, and a nonlinearity | 16 |
| 2 | Optimizers: SGD, momentum, Adam, AdaGrad | 16 |
| 3 | One training step | 8 |

### How it works

- This is an individual homework.
- Write code only between `### YOUR CODE HERE ###` and `### END YOUR CODE ###` (replace the `None` placeholders). Don’t edit the `print(...)` lines below each cell.
- There’s no auto-grader. Each cell’s own `print(...)` line shows you the real value your code produced, immediately followed by an inline `(expected: ...)` annotation. Run the cell and compare the two by eye — if they match, that sub-part is correct.
- **Nothing here is built from scratch.** Every problem calls PyTorch’s own layers, nonlinearities, and optimizers — `Sequential`, `BatchNorm1d`, `Dropout`, `tanh`, `SGD`, `Adam`, `Adagrad`, `cross_entropy` — the same calls used in (or, for AdaGrad, discussed in) the course slides.
- Dependencies: Problem 2’s test cells reuse `test_mlp` (built in Problem 1(a)). Problem 3(a)’s test cell builds its own `test_model2`, which the given `evaluate` right after it reuses. Run the problems in order.
- Every cell runs on **real data** — scikit-learn’s Breast Cancer Wisconsin dataset (569 patients, 30 real tumor measurements, malignant/benign), wrapped in a `DataLoader` for minibatches, exactly like the course’s own training loop.
- The homework ends with a given “Putting it together” cell (no blanks, not graded) that reuses everything you just built to compare SGD, momentum, Adam, and AdaGrad on the same patients. **Restart & Run All** before submitting.

### Setup (given — run once, nothing to fill in)

Loads the real dataset (same train/val split and standardization as before), wraps the training set in a `DataLoader` for minibatches, and builds the fixtures every problem below is tested against. You’ll build `MLP` and `apply_tanh` yourself, in Problem 1.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

torch.manual_seed(0)

# ----------------- real dataset: Breast Cancer Wisconsin -----------------
# 569 patients, 30 real tumor measurements, 2 classes (0=malignant, 1=benign).
data = load_breast_cancer()
X_all, y_all = data.data.astype("float32"), data.target.astype("int64")
_Xtr, _Xva, _ytr, _yva = train_test_split(
    X_all, y_all, test_size=0.2, random_state=0, stratify=y_all)
X_train_raw, X_val_raw = torch.tensor(_Xtr), torch.tensor(_Xva)
y_train, y_val = torch.tensor(_ytr), torch.tensor(_yva)

_mu, _sigma = X_train_raw.mean(0), X_train_raw.std(0, unbiased=False)
X_train = (X_train_raw - _mu) / _sigma      # standardized with TRAINING statistics only
X_val = (X_val_raw - _mu) / _sigma

num_features = X_train.shape[1]                # 30

# ----------------- minibatches, the course’s own way -----------------
train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
device = "cpu"

print(f"train: {tuple(X_train.shape)} val: {tuple(X_val.shape)} "
      f"pos rate train/val: {y_train.float().mean():.3f} / {y_val.float().mean():.3f}")


## Problem 1 — The regularized MLP, and a nonlinearity (16 points)

### 1(a) (12 pts) — MLP

Complete `MLP` below, combining two regularization techniques in one network: a `Sequential` of `Linear` (no bias) → `BatchNorm1d` → `ReLU` → `Dropout(p=0.3)` → `Linear`. Use `bias=False` on the first `Linear` layer — BatchNorm’s own learned shift already controls the mean, so that bias term would be redundant. Run the cell and compare your printed output, line by line, to the `(expected: ...)` value next to it.

In [ ]:
class MLP(torch.nn.Module):
    def __init__(self, num_features, num_hidden, num_classes):
        super().__init__()
        ### YOUR CODE HERE ###
        self.layers = torch.nn.Sequential(
            None,   # Linear: num_features -> num_hidden, no bias
            None,   # BatchNorm1d: num_hidden
            None,   # ReLU
            None,   # Dropout: p=0.3
            None,   # Linear: num_hidden -> num_classes
        )
        ### END YOUR CODE ###

    def forward(self, x):
        ### YOUR CODE HERE ###
        return None
        ### END YOUR CODE ###

torch.manual_seed(0)
test_mlp = MLP(num_features, 32, 2)
print("layer 0:", type(test_mlp.layers[0]).__name__, "| bias =",
      test_mlp.layers[0].bias, " (expected: Linear | bias = None)")
print("layer 1:", type(test_mlp.layers[1]).__name__,
      " (expected: BatchNorm1d)")
print("layer 3:", type(test_mlp.layers[3]).__name__, "| p =",
      test_mlp.layers[3].p, " (expected: Dropout | p = 0.3)")
test_mlp.train()
print("output shape on 4 real patients =",
      tuple(test_mlp(X_train[:4]).shape), " (expected: (4, 2))")


### 1(b) (4 pts) — apply_tanh

Implement `apply_tanh`, the other classic nonlinearity: $\tanh(z) = (e^z - e^{-z})/(e^z + e^{-z})$, squashing to $(-1, 1)$ instead of ReLU’s $[0, \infty)$. Call `torch.tanh` directly. (Tanh is also the activation that Xavier initialization assumes a centered, near-linear response around zero; ReLU breaks that assumption, which is why He initialization corrects for it instead.) Tested on 5 real patients’ first measurement.

In [ ]:
def apply_tanh(z):
    ### YOUR CODE HERE ###
    return None
    ### END YOUR CODE ###

z_test = X_train[:5, 0]
print("tanh(z) =", [round(v, 4) for v in apply_tanh(z_test).tolist()],
      " (expected: [0.2606, -0.9061, 0.9058, -0.8268, -0.5454])")


## Problem 2 — Optimizers: SGD, momentum, Adam, AdaGrad (16 points)

### 2(a) (4 pts) — build_sgd_optimizer

Write `build_sgd_optimizer`, which builds and returns a plain stochastic gradient descent optimizer for the given model using `torch.optim.SGD`. This is the bare update rule with no memory of past gradients — every parameter moves by the same fixed learning rate on every step.

In [ ]:
def build_sgd_optimizer(model, lr=0.01):
    ### YOUR CODE HERE ###
    return None
    ### END YOUR CODE ###

test_opt = build_sgd_optimizer(test_mlp)
print(type(test_opt).__name__, "| momentum =", test_opt.defaults['momentum'],
      " (expected: SGD | momentum = 0)")


### 2(b) (4 pts) — build_momentum_optimizer

Write `build_momentum_optimizer`, which builds and returns an SGD optimizer that also tracks momentum: pass `momentum=momentum` to `torch.optim.SGD` so each update also carries forward a fraction of the previous step’s direction, instead of reacting only to the current gradient.

In [ ]:
def build_momentum_optimizer(model, lr=0.01, momentum=0.9):
    ### YOUR CODE HERE ###
    return None
    ### END YOUR CODE ###

test_opt = build_momentum_optimizer(test_mlp)
print(type(test_opt).__name__, "| momentum =", test_opt.defaults['momentum'],
      " (expected: SGD | momentum = 0.9)")


### 2(c) (4 pts) — build_adam_optimizer

Write `build_adam_optimizer`, which builds and returns a bias-corrected Adam optimizer using `torch.optim.Adam` — Adam combines momentum with RMSProp’s per-parameter adaptive learning rates. Pass `weight_decay` straight through so the optimizer also applies L2 weight regularization during training.

In [ ]:
def build_adam_optimizer(model, lr=0.001, weight_decay=0.0):
    ### YOUR CODE HERE ###
    return None
    ### END YOUR CODE ###

test_opt = build_adam_optimizer(test_mlp, weight_decay=1e-4)
print(type(test_opt).__name__, "| lr =", test_opt.defaults['lr'],
      "| weight_decay =", test_opt.defaults['weight_decay'],
      " (expected: Adam | lr = 0.001 | weight_decay = 0.0001)")


### 2(d) (4 pts) — build_adagrad_optimizer

Build and return an AdaGrad optimizer via `torch.optim.Adagrad`. AdaGrad shrinks each parameter’s learning rate by its own running sum of squared gradients, so frequently-updated weights slow down fastest.

In [ ]:
def build_adagrad_optimizer(model, lr=0.01):
    ### YOUR CODE HERE ###
    return None
    ### END YOUR CODE ###

test_opt = build_adagrad_optimizer(test_mlp)
print(type(test_opt).__name__, "| lr =", test_opt.defaults['lr'],
      " (expected: Adagrad | lr = 0.01)")


## Problem 3 — One training step (8 points)

### 3(a) (8 pts) — train_one_epoch

One pass over `train_loader`: switch to training mode (so Dropout is active), then for every minibatch run the course’s three-call sequence — `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()`.

In [ ]:
def train_one_epoch(model, optimizer, loader):
    ### YOUR CODE HERE ###
    None                  # put the model in training mode
    ### END YOUR CODE ###
    for features, targets in loader:
        ### YOUR CODE HERE ###
        loss = None        # F.cross_entropy on model(features), targets
        None               # clear old gradients
        None               # backpropagate
        None               # take one optimizer step
        ### END YOUR CODE ###
    return loss

torch.manual_seed(0)
test_model2 = MLP(num_features, 32, 2)
test_opt2 = build_sgd_optimizer(test_model2, lr=0.01)
tiny_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_train[:64], y_train[:64]), batch_size=32)
last_loss = train_one_epoch(test_model2, test_opt2, tiny_loader)
print("last batch loss =", round(last_loss.item(), 4), " (expected: 0.7032)")


### (given — not an exercise) — evaluate

Validation accuracy, the straightforward way: `model.eval()` so Dropout turns off and BatchNorm switches to its running statistics, inside `torch.no_grad()` since you’re not backpropagating a metric. The predicted class is whichever logit is larger — `argmax(dim=1)`. You don’t write this one — it’s given here because `train_model` below needs it.

In [ ]:
def evaluate(model, X, y):
    model.eval()
    with torch.no_grad():
        preds = model(X).argmax(dim=1)
        acc = (preds == y).float().mean().item()
    return acc

val_acc = evaluate(test_model2, X_val, y_val)
print("val acc after 1 tiny epoch =", round(val_acc, 4))


## Putting it together (given — no blanks, not graded)

Nothing new to derive here — every line below just calls a function you already wrote and tested above. Twenty epochs each, on the same real patients, swapping only the optimizer.

In [ ]:
def train_model(name, optimizer_fn, num_epochs=20, seed=0):
    torch.manual_seed(seed)
    model = MLP(num_features, 32, 2).to(device)
    optimizer = optimizer_fn(model)
    best_val_acc, best_epoch = 0.0, -1
    for epoch in range(num_epochs):
        loss = train_one_epoch(model, optimizer, train_loader)
        val_acc = evaluate(model, X_val, y_val)
        if val_acc > best_val_acc:
            best_val_acc, best_epoch = val_acc, epoch + 1
    print(f"{name:<28} final train loss={loss.item():.3f} "
          f"best val acc={best_val_acc*100:5.1f}% (epoch {best_epoch:>2})")
    return best_val_acc

train_model("SGD (lr=0.01)", lambda m: build_sgd_optimizer(m, lr=0.01))
train_model("SGD + momentum=0.9",
            lambda m: build_momentum_optimizer(m, lr=0.01, momentum=0.9))
train_model("Adam (lr=0.001)", lambda m: build_adam_optimizer(m, lr=0.001))
train_model("Adam + weight_decay=1e-4",
            lambda m: build_adam_optimizer(m, lr=0.001, weight_decay=1e-4))
train_model("AdaGrad (lr=0.01)", lambda m: build_adagrad_optimizer(m, lr=0.01))
